# Otonom Sürüş PPO Müfredat Eğitimi (Sweet Spot v3)
Bu notebook, **Stable-Baselines3** kullanarak 6 aşamalı gelişmiş bir müfredat eğitimi (Curriculum Learning) ile otonom sürüş ajanını eğitir.

In [ ]:
# ==========================================
# 1. GEREKLİLİKLER VE KURULUM
# ==========================================
!pip install stable-baselines3[extra] gymnasium numpy torch -q

In [ ]:
import os
import math
import random
import time
from collections import deque
import numpy as np
import gymnasium as gym
from gymnasium import spaces

import torch
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import BaseCallback

## 2. Ortam (Environment) ve Destek Sınıfları
Ajanın etkileşime gireceği otonom sürüş simülasyon ortamı (`AutonomousDriverEnv`), dinamik engeller (`DynamicObstacle`) ve BFS tabanlı %100 çözülebilir harita üreteci (`MapGenerator`) bu bölümde tanımlanır.

In [ ]:
class DynamicObstacle:
    def __init__(self, x, y, vx, vy, grid_size):
        self.x, self.y = float(x), float(y)
        self.vx, self.vy = float(vx), float(vy)
        self.grid_size = grid_size
    def step(self):
        self.x += self.vx
        self.y += self.vy
        if self.x < 0 or self.x >= self.grid_size - 1: self.vx *= -1
        if self.y < 0 or self.y >= self.grid_size - 1: self.vy *= -1
    @property
    def grid_pos(self): return (int(round(self.x)), int(round(self.y)))
    def normalized_velocity(self, max_speed): return (self.vx / max_speed, self.vy / max_speed)

class MapGenerator:
    def __init__(self, grid_size, n_static):
        self.grid_size, self.n_static = grid_size, n_static

    def has_path(self, start, goal, static_obstacles):
        """BFS ile başlangıç ve hedef arasında en az bir yol olduğunu garanti eder."""
        queue = deque([start])
        visited = {start}
        while queue:
            curr = queue.popleft()
            if curr == goal:
                return True
            for dx, dy in [(0,1), (0,-1), (1,0), (-1,0)]:
                nx, ny = curr[0] + dx, curr[1] + dy
                if 0 <= nx < self.grid_size and 0 <= ny < self.grid_size:
                    if (nx, ny) not in static_obstacles and (nx, ny) not in visited:
                        visited.add((nx, ny))
                        queue.append((nx, ny))
        return False

    def generate(self, map_type=None):
        static_obs = set()
        while len(static_obs) < self.n_static:
            static_obs.add((random.randint(0, self.grid_size-1), random.randint(0, self.grid_size-1)))
        return static_obs, "procedural"

    def get_start_and_goal(self, static_obstacles):
        """%100 çözülebilir bir başlangıç ve hedef ikilisi bulana kadar dener."""
        attempts = 0
        while True:
            attempts += 1
            s = (random.randint(0, self.grid_size-1), random.randint(0, self.grid_size-1))
            g = (random.randint(0, self.grid_size-1), random.randint(0, self.grid_size-1))
            if s not in static_obstacles and g not in static_obstacles and s != g:
                if math.hypot(s[0]-g[0], s[1]-g[1]) > self.grid_size * 0.4:
                    # BFS Yol Kontrolü: Yol yoksa haritayı yeniden dene
                    if self.has_path(s, g, static_obstacles):
                        return s, g
            # Eğer 50 denemede yol çıkmazsa statik engelleri hafifçe azaltıp dene (kilitlenmeyi önler)
            if attempts > 50:
                static_obstacles.clear()
                while len(static_obstacles) < self.n_static:
                    static_obstacles.add((random.randint(0, self.grid_size-1), random.randint(0, self.grid_size-1)))
                attempts = 0

class AutonomousDriverEnv(gym.Env):
    def __init__(self, grid_size=12, n_static=4, n_dynamic=0, max_steps=150, view_radius=7, render_mode=None):
        super(AutonomousDriverEnv, self).__init__()
        self.grid_size = grid_size
        self.n_static = n_static
        self.n_dynamic = n_dynamic
        self.max_steps = max_steps
        self.view_radius = view_radius
        self.render_mode = render_mode
        self.map_gen = MapGenerator(grid_size, n_static)

        self.action_space = spaces.Discrete(5)
        self.observation_space = spaces.Box(low=-1.0, high=1.0, shape=(102,), dtype=np.float32)

        self.agent_pos = None
        self.goal_pos = None
        self.static_obs = set()
        self.dynamic_obs = []
        self.visit_map = None
        self.action_history = None
        self.steps = 0
        self.prev_dist = 0
        self.map_type_current = ""
        self.agent_trajectory = []

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.static_obs, self.map_type_current = self.map_gen.generate()
        self.agent_pos, self.goal_pos = self.map_gen.get_start_and_goal(self.static_obs)
        self.agent_trajectory = [self.agent_pos]

        self.dynamic_obs = []
        for _ in range(self.n_dynamic):
            while True:
                dx = random.randint(0, self.grid_size-1)
                dy = random.randint(0, self.grid_size-1)
                if (dx, dy) not in self.static_obs and (dx, dy) != self.agent_pos and (dx, dy) != self.goal_pos:
                    speed = 0.5
                    if random.choice([True, False]):
                        vx = random.choice([-speed, speed])
                        vy = 0.0
                    else:
                        vx = 0.0
                        vy = random.choice([-speed, speed])
                    self.dynamic_obs.append(DynamicObstacle(dx, dy, vx, vy, self.grid_size))
                    break

        self.visit_map = np.zeros((self.grid_size, self.grid_size), dtype=np.float32)
        self.visit_map[self.agent_pos[0], self.agent_pos[1]] += 1

        self.action_history = deque([[0.0]*5 for _ in range(8)], maxlen=8)
        self.steps = 0
        self.prev_dist = math.hypot(self.agent_pos[0]-self.goal_pos[0], self.agent_pos[1]-self.goal_pos[1])

        info = {"map_type": self.map_type_current}
        return self._get_obs(), info

    def step(self, action):
        self.steps += 1
        reward = -0.05
        terminated = False
        truncated = False
        is_success = False

        action_oh = [0.0]*5
        action_oh[action] = 1.0
        self.action_history.append(action_oh)

        for dyn in self.dynamic_obs:
            dyn.step()

        dx, dy = 0, 0
        if action == 0: dy = -1
        elif action == 1: dy = 1
        elif action == 2: dx = -1
        elif action == 3: dx = 1

        move_attempted = (dx != 0 or dy != 0)
        is_waiting = (action == 4)

        nx, ny = self.agent_pos[0] + dx, self.agent_pos[1] + dy

        hit_static = False
        if move_attempted:
            if not (0 <= nx < self.grid_size and 0 <= ny < self.grid_size):
                hit_static = True
            elif (nx, ny) in self.static_obs:
                hit_static = True

        if hit_static:
            reward -= 50.0
            terminated = True
            self.visit_map[self.agent_pos[0], self.agent_pos[1]] += 1
        elif move_attempted:
            self.agent_pos = (nx, ny)
            visits = self.visit_map[nx, ny]
            if visits == 0:
                reward += 0.3
            else:
                reward -= 0.05 * visits
            self.visit_map[nx, ny] += 1
            self.agent_trajectory.append(self.agent_pos)
        elif is_waiting:
            pass

        if not terminated:
            min_dyn_dist = float('inf')
            for dyn in self.dynamic_obs:
                dist = math.hypot(self.agent_pos[0] - dyn.x, self.agent_pos[1] - dyn.y)
                if dist < min_dyn_dist:
                    min_dyn_dist = dist

            if min_dyn_dist <= 0.8:
                reward -= 50.0
                terminated = True
            elif min_dyn_dist <= 1.5:
                reward -= 0.2

        curr_dist = math.hypot(self.agent_pos[0]-self.goal_pos[0], self.agent_pos[1]-self.goal_pos[1])
        reward += (self.prev_dist - curr_dist) * 1.5
        self.prev_dist = curr_dist

        if not terminated and self.agent_pos == self.goal_pos:
            reward += 100.0
            terminated = True
            is_success = True

        if not terminated and self.steps >= self.max_steps:
            truncated = True

        info = {
            "is_success": is_success,
            "map_type": self.map_type_current
        }
        return self._get_obs(), float(reward), terminated, truncated, info

    def _get_obs(self):
        obs = []
        dirs = [(0,-1), (1,-1), (1,0), (1,1), (0,1), (-1,1), (-1,0), (-1,-1)]
        dyn_positions_dict = {dyn.grid_pos: dyn for dyn in self.dynamic_obs}

        for dx, dy in dirs:
            hit_data = [1.0, 0.0, 0.0, 0.0]
            for step in range(1, self.view_radius + 1):
                rx, ry = self.agent_pos[0] + dx*step, self.agent_pos[1] + dy*step
                if rx < 0 or rx >= self.grid_size or ry < 0 or ry >= self.grid_size or (rx, ry) in self.static_obs:
                    hit_data = [step/self.view_radius, 1.0, 0.0, 0.0]
                    break
                if (rx, ry) in dyn_positions_dict:
                    dyn = dyn_positions_dict[(rx, ry)]
                    nvx, nvy = dyn.normalized_velocity(1.0)
                    hit_data = [step/self.view_radius, 0.5, nvx, nvy]
                    break
            obs.extend(hit_data)

        delta_x = (self.goal_pos[0] - self.agent_pos[0]) / self.grid_size
        delta_y = (self.goal_pos[1] - self.agent_pos[1]) / self.grid_size
        dist_norm = math.hypot(delta_x, delta_y) / math.sqrt(2)
        angle = math.atan2(delta_y, delta_x)
        obs.extend([delta_x, delta_y, dist_norm, math.sin(angle), math.cos(angle)])

        max_vis = max(1.0, np.max(self.visit_map))
        for dy in range(-2, 3):
            for dx in range(-2, 3):
                px, py = self.agent_pos[0]+dx, self.agent_pos[1]+dy
                if 0 <= px < self.grid_size and 0 <= py < self.grid_size:
                    obs.append(self.visit_map[px, py] / max_vis)
                else:
                    obs.append(1.0)

        for act_arr in self.action_history:
            obs.extend(act_arr)

        return np.array(obs, dtype=np.float32)

## 3. Rich Eğitim Takip Callback Sınıfı
Eğitim sırasında gürültüsüz, temiz ve detaylı metrik takibi sağlayan değerlendirme callback sınıfı (`RichTrainingCallback`).

In [ ]:
class RichTrainingCallback(BaseCallback):
    def __init__(self, eval_env, eval_freq=2048, n_eval=20, verbose=1):
        super().__init__(verbose)
        self.eval_env = eval_env
        self.eval_freq = eval_freq
        self.n_eval = n_eval
        self.log_steps = []
        self.log_rewards = []
        self.log_success_rate = []

    def _on_step(self) -> bool:
        if self.n_calls % self.eval_freq == 0:
            rewards = []
            success_count = 0
            for _ in range(self.n_eval):
                obs, info = self.eval_env.reset()
                done = False
                ep_reward = 0
                while not done:
                    action, _ = self.model.predict(obs, deterministic=True)
                    obs, reward, terminated, truncated, info = self.eval_env.step(action)
                    ep_reward += float(reward)
                    done = terminated or truncated
                rewards.append(ep_reward)
                if info.get("is_success", False):
                    success_count += 1
            
            mean_reward = np.mean(rewards)
            success_rate = success_count / self.n_eval
            self.log_steps.append(self.num_timesteps)
            self.log_rewards.append(mean_reward)
            self.log_success_rate.append(success_rate)
            
            print(f"Adım: {self.num_timesteps:7d} | Değerlendirme Ödülü: {mean_reward:6.2f} | Başarı Oranı: %{success_rate*100:5.1f}")
        return True

## 4. Gelişmiş 6 Aşamalı Ultimate Müfredat Eğitimi
Ajanı sırasıyla engelsiz haritadan hardcore dinamik engelli haritaya kadar 6 farklı aşamada (toplam 1.8 Milyon adım) eğitir.

In [ ]:
def train_super_model(save_path="./autonomous_driver"):
    os.makedirs(save_path, exist_ok=True)

    # 1.8 Milyon Adımlık Zirve Müfredat
    stages = [
        # Aşama 0: Engelsiz Başlangıç (Yön bulmayı 100% çözme)
        {"grid_size": 10, "n_static": 0,  "n_dynamic": 0, "max_steps": 80,  "timesteps": 100_000, "lr": 3e-4},
        # Aşama 1: Isınma (Statik engellerin etrafından dolanmayı öğrenme)
        {"grid_size": 12, "n_static": 5,  "n_dynamic": 0, "max_steps": 150, "timesteps": 150_000, "lr": 3e-4},
        # Aşama 2: Kolay Seviye (İlk dinamik engellerle tanışma)
        {"grid_size": 12, "n_static": 5,  "n_dynamic": 2, "max_steps": 180, "timesteps": 250_000, "lr": 2e-4},
        # Aşama 3: Orta Seviye (Ana harita çözünürlüğü ve yoğunluğu)
        {"grid_size": 15, "n_static": 12, "n_dynamic": 4, "max_steps": 250, "timesteps": 350_000, "lr": 1.5e-4},
        # Aşama 4: Zor Seviye (Büyük alanda karmaşık kaçışlar)
        {"grid_size": 18, "n_static": 20, "n_dynamic": 6, "max_steps": 350, "timesteps": 450_000, "lr": 1e-4},
        # Aşama 5: Hardcore Sınırları (Tehlike anında sabırla bekleme ve sızma)
        {"grid_size": 20, "n_static": 30, "n_dynamic": 8, "max_steps": 400, "timesteps": 500_000, "lr": 5e-5}
    ]

    policy_kwargs = dict(net_arch=[256, 256, 128])
    model = None

    for i, stage in enumerate(stages):
        print(f"\n🔥 BAŞLIYOR: Aşama {i} (Grid: {stage['grid_size']}x{stage['grid_size']} | Statik: {stage['n_static']} | Dinamik: {stage['n_dynamic']})")
        print(f"Öğrenme Oranı (LR): {stage['lr']} | Adım Limit: {stage['max_steps']} | Toplam Adım: {stage['timesteps']}")

        env_config = {
            "grid_size": stage['grid_size'],
            "n_static": stage['n_static'],
            "n_dynamic": stage['n_dynamic'],
            "max_steps": stage['max_steps'],
            "view_radius": 4  # Sweet Spot (102-dimension compatible)
        }

        env = DummyVecEnv([lambda: AutonomousDriverEnv(**env_config)])
        eval_env = AutonomousDriverEnv(**env_config)
        callback = RichTrainingCallback(eval_env, eval_freq=2048, n_eval=20)

        if model is None:
            model = PPO("MlpPolicy", env, policy_kwargs=policy_kwargs,
                        learning_rate=stage['lr'], n_steps=2048, batch_size=256,
                        n_epochs=10, gamma=0.99, gae_lambda=0.95,
                        clip_range=0.2, ent_coef=0.015, verbose=0)
        else:
            model.set_env(env)
            # PyTorch optimizer seviyesinde LR'ı güncelliyoruz:
            model.learning_rate = stage['lr']
            for param_group in model.policy.optimizer.param_groups:
                param_group['lr'] = stage['lr']

        model.learn(total_timesteps=stage["timesteps"], callback=callback)
        
        # Her aşama sonunda modeli kaydet
        model.save(f"{save_path}/ppo_sweetspot")
        print(f"✨ Aşama {i} tamamlandı. Model kaydedildi.")

    print("\n🎉 TEBRİKLER! TÜM SWEET SPOT MÜFREDAT BAŞARIYLA TAMAMLANDI!")
    return model

train_super_model()